# Hey Banco — Datathon 2026
## EDA UC4: Inteligencia Conversacional — dataset_50k_anonymized

**Objetivo:** Explorar las 49,999 interacciones entre usuarios y Havi para entender qué preguntan, qué problemas reportan y cómo responde el asistente. Este análisis valida que los 4 UCs son problemas reales que los usuarios ya están reportando.

| Sección | Descripción |
|---------|-------------|
| 0 | Setup y carga de datos |
| 1 | Schema, calidad y métricas base |
| 2 | Distribución canal (texto vs voz) |
| 3 | Turnos por conversación |
| 4 | Longitud de mensajes (input / output) |
| 5 | Top 20 palabras + WordCloud |
| 6 | Clasificación manual de intenciones (muestra 50) |
| 7 | Cruce user_id → hey_clientes (perfil vs consulta) |
| 8 | Análisis temporal (hora del día) |

---
**Dataset:** `dataset_50k_anonymized.parquet` · **Autores:** Diego Quiros (EDA) · Fernando Haro (UC4 FE & Modelado)

## 0. Setup y carga de datos

In [ ]:
import subprocess, sys
for pkg in ["pyarrow", "wordcloud"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import warnings
warnings.filterwarnings("ignore")

import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from wordcloud import WordCloud, STOPWORDS

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 100)

# ── Paleta del proyecto ───────────────────────────────────────────────────────
PALETTE = {
    'primary':   '#E63946',   # rojo Hey
    'secondary': '#457B9D',   # azul
    'accent':    '#F4A261',   # naranja
    'neutral':   '#A8DADC',   # celeste
    'bg':        '#F8F9FA',
    'text':      '#1D3557',
}

# ── Rutas ─────────────────────────────────────────────────────────────────────
BASE_TXN  = Path(r"Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_CONV = Path(r"Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones")

print("BASE_CONV:", BASE_CONV.resolve())
print("BASE_TXN: ", BASE_TXN.resolve())

In [ ]:
# ── Cargar datasets ───────────────────────────────────────────────────────────
df_convs = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")
df_convs["date"] = pd.to_datetime(df_convs["date"], format="mixed")

df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})

print(f"df_convs   : {df_convs.shape[0]:,} filas × {df_convs.shape[1]} cols")
print(f"df_clientes: {df_clientes.shape[0]:,} filas × {df_clientes.shape[1]} cols")

## 1. Schema, calidad y métricas base

In [ ]:
print("=" * 60)
print("SCHEMA — df_convs")
print("=" * 60)
print(df_convs.dtypes.to_string())
print()
print("=" * 60)
print("PRIMERAS 3 FILAS")
print("=" * 60)
df_convs.head(3)

In [ ]:
# ── Nulos y cardinalidad ──────────────────────────────────────────────────────
quality = pd.DataFrame({
    "dtype":       df_convs.dtypes,
    "nulos":       df_convs.isna().sum(),
    "nulos_%":     (df_convs.isna().mean() * 100).round(2),
    "únicos":      df_convs.nunique(),
})
print(quality.to_string())

In [ ]:
# ── Métricas base del dataset ─────────────────────────────────────────────────
n_turnos      = len(df_convs)
n_convs       = df_convs["conv_id"].nunique()
n_users       = df_convs["user_id"].nunique()
fecha_min     = df_convs["date"].min()
fecha_max     = df_convs["date"].max()
media_turnos  = n_turnos / n_convs

print("─" * 45)
print(f"  Turnos totales          : {n_turnos:>8,}")
print(f"  Conversaciones únicas   : {n_convs:>8,}")
print(f"  Usuarios únicos         : {n_users:>8,}")
print(f"  Turnos / conv (media)   : {media_turnos:>8.2f}")
print(f"  Período                 : {fecha_min.date()} → {fecha_max.date()}")
print("─" * 45)

## 2. Distribución por canal (texto vs voz)

In [ ]:
# channel_source es string "1" (texto) o "2" (voz)
canal_map = {"1": "Texto", "2": "Voz"}
df_convs["canal_label"] = df_convs["channel_source"].astype(str).map(canal_map).fillna("Otro")

canal_counts = df_convs["canal_label"].value_counts()
canal_pct    = (canal_counts / len(df_convs) * 100).round(1)

print("Canal de contacto:")
for label in canal_counts.index:
    print(f"  {label:10}: {canal_counts[label]:>6,}  ({canal_pct[label]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor=PALETTE['bg'])

# Barra
ax = axes[0]
colors_bar = [PALETTE['secondary'], PALETTE['accent']]
bars = ax.bar(canal_counts.index, canal_counts.values, color=colors_bar[:len(canal_counts)], width=0.5)
for bar, pct in zip(bars, canal_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
            f"{pct:.1f}%", ha='center', va='bottom', fontsize=11, fontweight='bold',
            color=PALETTE['text'])
ax.set_title("Interacciones por Canal", fontsize=13, color=PALETTE['text'], pad=10)
ax.set_ylabel("N° de interacciones", color=PALETTE['text'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_facecolor(PALETTE['bg'])
ax.spines[['top','right']].set_visible(False)
ax.tick_params(colors=PALETTE['text'])

# Pie
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    canal_counts.values,
    labels=canal_counts.index,
    autopct='%1.1f%%',
    colors=colors_bar[:len(canal_counts)],
    startangle=90,
    pctdistance=0.75,
    textprops={'color': PALETTE['text'], 'fontsize': 11}
)
for at in autotexts:
    at.set_fontweight('bold')
ax2.set_title("Proporción Canal", fontsize=13, color=PALETTE['text'], pad=10)

fig.suptitle("UC4 — Canal de Contacto con Havi", fontsize=14, fontweight='bold',
             color=PALETTE['text'], y=1.02)
plt.tight_layout()
plt.savefig("uc4_fig1_canal.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig1_canal.png")

## 3. Distribución de turnos por conversación

In [ ]:
turns_per_conv = df_convs.groupby("conv_id").size().rename("n_turnos")

desc = turns_per_conv.describe(percentiles=[.5, .75, .90, .95, .99])
print("Distribución de turnos por conversación:")
print(desc.to_string())
print()
print("Top frecuencias (N turnos → N conversaciones):")
print(turns_per_conv.value_counts().sort_index().head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor=PALETTE['bg'])

# Histograma turnos (acotado a p99)
p99 = int(turns_per_conv.quantile(0.99))
ax = axes[0]
ax.hist(turns_per_conv.clip(upper=p99), bins=range(1, p99 + 2),
        color=PALETTE['secondary'], edgecolor='white', linewidth=0.5)
ax.axvline(turns_per_conv.mean(), color=PALETTE['primary'], lw=1.8, linestyle='--',
           label=f"Media {turns_per_conv.mean():.1f}")
ax.axvline(turns_per_conv.median(), color=PALETTE['accent'], lw=1.8, linestyle=':',
           label=f"Mediana {turns_per_conv.median():.0f}")
ax.legend(fontsize=9)
ax.set_title("Histograma: turnos por conv (hasta p99)", fontsize=12, color=PALETTE['text'])
ax.set_xlabel("N° de turnos", color=PALETTE['text'])
ax.set_ylabel("N° de conversaciones", color=PALETTE['text'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_facecolor(PALETTE['bg'])
ax.spines[['top','right']].set_visible(False)

# CDF acumulada
ax2 = axes[1]
sorted_vals = np.sort(turns_per_conv.values)
cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
ax2.plot(sorted_vals, cdf, color=PALETTE['secondary'], lw=2)
for pct, ls in [(0.5, ':'), (0.9, '--'), (0.99, '-.')]:  
    v = int(turns_per_conv.quantile(pct))
    ax2.axvline(v, color=PALETTE['primary'], lw=1, linestyle=ls,
                label=f"p{int(pct*100)}={v}")
ax2.set_xlim(0, max(30, p99))
ax2.legend(fontsize=9)
ax2.set_title("CDF acumulada de turnos por conv", fontsize=12, color=PALETTE['text'])
ax2.set_xlabel("N° de turnos", color=PALETTE['text'])
ax2.set_ylabel("Fracción acumulada", color=PALETTE['text'])
ax2.set_facecolor(PALETTE['bg'])
ax2.spines[['top','right']].set_visible(False)

fig.suptitle("UC4 — Distribución de Turnos por Conversación", fontsize=14, fontweight='bold',
             color=PALETTE['text'], y=1.02)
plt.tight_layout()
plt.savefig("uc4_fig2_turnos.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig2_turnos.png")

## 4. Longitud de mensajes: input y output

In [ ]:
# Aproximación de tokens: palabras separadas por espacios (no se requiere tokenizador)
df_convs["input_chars"]   = df_convs["input"].fillna("").str.len()
df_convs["output_chars"]  = df_convs["output"].fillna("").str.len()
df_convs["input_tokens"]  = df_convs["input"].fillna("").str.split().str.len()
df_convs["output_tokens"] = df_convs["output"].fillna("").str.split().str.len()

for col, label in [("input_tokens", "Input"), ("output_tokens", "Output")]:
    s = df_convs[col]
    print(f"\n── {label} (palabras / tokens) ──")
    print(f"  Media   : {s.mean():.1f}")
    print(f"  Mediana : {s.median():.0f}")
    print(f"  p95     : {s.quantile(.95):.0f}")
    print(f"  Máximo  : {s.max()}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), facecolor=PALETTE['bg'])

specs = [
    ("input_tokens",  "Input — palabras",   PALETTE['secondary'], axes[0, 0]),
    ("output_tokens", "Output — palabras",  PALETTE['accent'],    axes[0, 1]),
    ("input_chars",   "Input — caracteres", PALETTE['secondary'], axes[1, 0]),
    ("output_chars",  "Output — caracteres",PALETTE['accent'],    axes[1, 1]),
]

for col, title, color, ax in specs:
    p99 = df_convs[col].quantile(0.99)
    data = df_convs[col].clip(upper=p99)
    ax.hist(data, bins=50, color=color, edgecolor='white', linewidth=0.3)
    ax.axvline(df_convs[col].mean(), color=PALETTE['primary'], lw=1.5,
               linestyle='--', label=f"μ={df_convs[col].mean():.0f}")
    ax.legend(fontsize=8)
    ax.set_title(title, fontsize=11, color=PALETTE['text'])
    ax.set_xlabel("Longitud", color=PALETTE['text'])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.set_facecolor(PALETTE['bg'])
    ax.spines[['top','right']].set_visible(False)

fig.suptitle("UC4 — Longitud de Mensajes Input / Output", fontsize=14, fontweight='bold',
             color=PALETTE['text'])
plt.tight_layout()
plt.savefig("uc4_fig3_longitud.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig3_longitud.png")

## 5. Top 20 palabras más frecuentes en input + WordCloud

In [ ]:
# ── Stopwords en español ──────────────────────────────────────────────────────
STOPWORDS_ES = set("""
de la el los las un una unos unas y o a en que es se del con por para
al lo su sus mi tu su me te le nos les hay si no qué cómo dónde cuándo
cuánto quién más pero como bien ya también desde hasta hace sobre sin
entre cuando donde quien porque pues aunque ni sí así muy tanto tan
todo toda todos todas este esta estos estas ese esa esos esas aquel
aquella aquellos aquellas ser estar sido estoy estás está estamos estáis
están tengo tienes tiene tenemos tenéis tienen fue fui fuiste fueron
hay haber había he has ha hemos habéis han puedo puede podemos pueden
quiero quieres quiere queremos quieren hacer hago hace hacemos hacen
soy eres somos son era eran iba ivan iban hacia mismo misma mismos
mismas otro otra otros otras saber sé sabes sabe sabemos saben
hola hey hola buenas gracias favor porfavor por favor ok okey
""".split())

def normalize_text(text: str) -> str:
    """Lowercase, quitar acentos, quitar no-alfanumérico."""
    text = text.lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text

all_words = (
    df_convs["input"]
    .dropna()
    .apply(normalize_text)
    .str.split()
    .explode()
)
all_words = all_words[~all_words.isin(STOPWORDS_ES) & (all_words.str.len() > 2)]

word_freq = Counter(all_words)
top20 = word_freq.most_common(20)

print("Top 20 palabras más frecuentes en input (excl. stopwords):")
for rank, (word, count) in enumerate(top20, 1):
    print(f"  {rank:>2}. {word:<20} {count:>6,}")

In [ ]:
fig = plt.figure(figsize=(16, 6), facecolor=PALETTE['bg'])
gs = GridSpec(1, 2, figure=fig, width_ratios=[1, 1.3])

# ── Barras Top 20 ─────────────────────────────────────────────────────────────
ax_bar = fig.add_subplot(gs[0])
words_list  = [w for w, _ in top20]
counts_list = [c for _, c in top20]
colors_grad = [PALETTE['primary'] if i < 5 else PALETTE['secondary'] for i in range(20)]
bars = ax_bar.barh(words_list[::-1], counts_list[::-1], color=colors_grad[::-1])
ax_bar.set_title("Top 20 Palabras — Input", fontsize=12, color=PALETTE['text'])
ax_bar.set_xlabel("Frecuencia", color=PALETTE['text'])
ax_bar.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax_bar.set_facecolor(PALETTE['bg'])
ax_bar.spines[['top','right']].set_visible(False)
ax_bar.tick_params(axis='y', labelsize=9)

# ── WordCloud ─────────────────────────────────────────────────────────────────
ax_wc = fig.add_subplot(gs[1])
wc = WordCloud(
    width=700, height=380,
    background_color=PALETTE['bg'],
    colormap='RdYlBu_r',
    max_words=80,
    stopwords=STOPWORDS_ES,
    prefer_horizontal=0.85,
    min_font_size=8
).generate_from_frequencies(word_freq)
ax_wc.imshow(wc, interpolation='bilinear')
ax_wc.axis('off')
ax_wc.set_title("WordCloud — Input usuarios", fontsize=12, color=PALETTE['text'])
ax_wc.set_facecolor(PALETTE['bg'])

fig.suptitle("UC4 — Vocabulario de los Usuarios en Conversaciones con Havi",
             fontsize=14, fontweight='bold', color=PALETTE['text'])
plt.tight_layout()
plt.savefig("uc4_fig4_palabras.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig4_palabras.png")

## 6. Clasificación manual de intenciones (muestra 50)

Taxonomía de intenciones (basada en inspección del vocabulario y lectura de mensajes):

| Intent | Descripción |
|--------|-------------|
| `consulta_saldo` | Preguntas sobre saldo, cuenta, disponible |
| `problema_transaccion` | Cargos no reconocidos, transacción fallida, rechazos |
| `reporte_fraude` | Fraude, tarjeta clonada, movimiento sospechoso |
| `pregunta_beneficios` | Cashback, Hey Pro, beneficios, recompensas |
| `solicitud_producto` | Crédito, préstamo, tarjeta nueva, cuenta de ahorro |
| `soporte_tecnico` | App caída, no puedo entrar, error en plataforma |
| `otro` | Saludos, consultas genéricas, temas no financieros |

In [ ]:
# ── Muestra estratificada de 50 mensajes para clasificación ───────────────────
rng = np.random.default_rng(42)
sample_idx = rng.choice(df_convs.index, size=50, replace=False)
df_sample = df_convs.loc[sample_idx, ["input", "output", "channel_source"]].copy()
df_sample = df_sample.reset_index(drop=True)

# ── Clasificador de intenciones basado en palabras clave ─────────────────────
INTENT_RULES = [
    ("reporte_fraude",       r"fraude|clonada|robo|robaron|no reconoc|no autoriz|cargo extrano"),
    ("problema_transaccion", r"rechazo|rechaz|no proceso|no se proceso|transacci[oó]n fallo|cargo doble|devoluci[oó]n|reversion|no aparece|no se reflej"),
    ("consulta_saldo",       r"saldo|cuenta|disponible|balance|cuanto tengo|cuanto hay|dinero"),
    ("pregunta_beneficios",  r"cashback|beneficio|hey pro|recompensa|puntos|descuento|promo"),
    ("solicitud_producto",   r"cr[eé]dito|pr[eé]stamo|tarjeta|cuenta de ahorro|apertura|quiero contratar|solicitar"),
    ("soporte_tecnico",      r"app|no puedo entrar|error|no funciona|no carga|se cayó|plataforma|contrase[nñ]a|acceso"),
]

def classify_intent(text: str) -> str:
    if pd.isna(text):
        return "otro"
    t = normalize_text(text)
    for intent, pattern in INTENT_RULES:
        if re.search(pattern, t):
            return intent
    return "otro"

df_sample["intent"] = df_sample["input"].apply(classify_intent)

print("Distribución de intenciones en la muestra de 50:")
print(df_sample["intent"].value_counts().to_string())
print()
print("Muestra con intenciones clasificadas:")
df_sample[["intent", "input"]].head(20)

In [ ]:
# ── Aplicar clasificador al dataset completo (señal aproximada) ───────────────
df_convs["intent"] = df_convs["input"].apply(classify_intent)

intent_counts = df_convs["intent"].value_counts()
intent_pct    = (intent_counts / len(df_convs) * 100).round(1)

print("Distribución de intenciones — dataset completo:")
for intent in intent_counts.index:
    bar = "█" * int(intent_pct[intent])
    print(f"  {intent:<25}: {intent_counts[intent]:>6,}  ({intent_pct[intent]:.1f}%)  {bar}")

In [ ]:
INTENT_COLORS = {
    "consulta_saldo":        PALETTE['secondary'],
    "problema_transaccion":  PALETTE['primary'],
    "reporte_fraude":        '#C77DFF',
    "pregunta_beneficios":   PALETTE['neutral'],
    "solicitud_producto":    PALETTE['accent'],
    "soporte_tecnico":       '#52B788',
    "otro":                  '#ADB5BD',
}

fig, ax = plt.subplots(figsize=(10, 5), facecolor=PALETTE['bg'])
bar_colors = [INTENT_COLORS.get(i, '#ADB5BD') for i in intent_counts.index]
bars = ax.barh(intent_counts.index[::-1], intent_counts.values[::-1],
               color=bar_colors[::-1])
for bar, pct in zip(bars, intent_pct.values[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{pct:.1f}%", va='center', fontsize=9, color=PALETTE['text'])
ax.set_title("Distribución de Intenciones en Conversaciones con Havi",
             fontsize=13, fontweight='bold', color=PALETTE['text'])
ax.set_xlabel("N° de interacciones", color=PALETTE['text'])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_facecolor(PALETTE['bg'])
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("uc4_fig5_intents.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig5_intents.png")

### 6.1 Ejemplos de mensajes por intención

In [ ]:
for intent in intent_counts.index:
    subset = df_convs[df_convs["intent"] == intent]
    sample = subset.sample(min(3, len(subset)), random_state=42)
    print(f"\n{'─'*60}")
    print(f"  Intent: {intent.upper()} (N={len(subset):,})")
    print(f"{'─'*60}")
    for _, row in sample.iterrows():
        inp = str(row['input'])[:120].replace('\n', ' ')
        out = str(row['output'])[:120].replace('\n', ' ')
        print(f"  USER: {inp}")
        print(f"  HAVI: {out}")
        print()

## 7. Cruce user_id → hey_clientes (perfil vs tipo de consulta)

In [ ]:
# ── Join conversaciones + clientes ────────────────────────────────────────────
convs_enriquecido = df_convs.merge(df_clientes, on="user_id", how="left")

print(f"Conversaciones con perfil de cliente: {convs_enriquecido['edad'].notna().sum():,} / {len(convs_enriquecido):,}")
print(f"Usuarios únicos en conversaciones  : {df_convs['user_id'].nunique():,}")
print(f"Usuarios únicos en hey_clientes    : {df_clientes['user_id'].nunique():,}")

# Cobertura del join
users_convs   = set(df_convs["user_id"])
users_clientes = set(df_clientes["user_id"])
overlap = len(users_convs & users_clientes)
print(f"\nUsuarios presentes en ambos datasets: {overlap:,} ({overlap/len(users_convs)*100:.1f}% de convs, "
      f"{overlap/len(users_clientes)*100:.1f}% de clientes)")

In [ ]:
# ── Distribución de intenciones por segmento ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=PALETTE['bg'])

# 1) Intent por Hey Pro vs no-Pro
ax = axes[0]
ct = pd.crosstab(convs_enriquecido["es_hey_pro"],
                 convs_enriquecido["intent"], normalize='index') * 100
ct.plot(kind='bar', ax=ax, color=list(INTENT_COLORS.values())[:len(ct.columns)],
        width=0.6, legend=False)
ax.set_title("Intent por Hey Pro", fontsize=11, color=PALETTE['text'])
ax.set_xlabel("Hey Pro", color=PALETTE['text'])
ax.set_ylabel("%", color=PALETTE['text'])
ax.set_xticklabels(["No Pro", "Pro"], rotation=0)
ax.set_facecolor(PALETTE['bg'])
ax.spines[['top','right']].set_visible(False)

# 2) Intent por canal (texto vs voz)
ax2 = axes[1]
ct2 = pd.crosstab(convs_enriquecido["canal_label"],
                  convs_enriquecido["intent"], normalize='index') * 100
ct2.plot(kind='bar', ax=ax2, color=list(INTENT_COLORS.values())[:len(ct2.columns)],
         width=0.6, legend=True, legend_labels=ct2.columns.tolist())
ax2.set_title("Intent por Canal", fontsize=11, color=PALETTE['text'])
ax2.set_xlabel("Canal", color=PALETTE['text'])
ax2.set_ylabel("%", color=PALETTE['text'])
ax2.set_xticklabels(ct2.index, rotation=0)
ax2.legend(fontsize=7, loc='upper right')
ax2.set_facecolor(PALETTE['bg'])
ax2.spines[['top','right']].set_visible(False)

# 3) Conversaciones por edad (decil)
ax3 = axes[2]
convs_enriquecido["edad_bin"] = pd.cut(
    convs_enriquecido["edad"], bins=[17, 25, 30, 35, 40, 50, 61],
    labels=["18-25", "26-30", "31-35", "36-40", "41-50", "51+"]
)
edad_intent = pd.crosstab(convs_enriquecido["edad_bin"],
                           convs_enriquecido["intent"], normalize='index') * 100
edad_intent.plot(kind='bar', ax=ax3, stacked=True,
                 color=list(INTENT_COLORS.values())[:len(edad_intent.columns)],
                 width=0.7, legend=False)
ax3.set_title("Intent por Rango de Edad", fontsize=11, color=PALETTE['text'])
ax3.set_xlabel("Edad", color=PALETTE['text'])
ax3.set_ylabel("%", color=PALETTE['text'])
ax3.set_xticklabels(edad_intent.index, rotation=30)
ax3.set_facecolor(PALETTE['bg'])
ax3.spines[['top','right']].set_visible(False)

fig.suptitle("UC4 — Intenciones por Segmento de Usuario", fontsize=14, fontweight='bold',
             color=PALETTE['text'])
plt.tight_layout()
plt.savefig("uc4_fig6_segmentos.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig6_segmentos.png")

In [ ]:
# ── ¿Los usuarios con patrón atípico contactan más por fraude? ────────────────
fraud_atipico = convs_enriquecido.groupby(["patron_uso_atipico", "intent"]).size().unstack(fill_value=0)
fraud_atipico_pct = fraud_atipico.div(fraud_atipico.sum(axis=1), axis=0) * 100

print("\nDistribución de intenciones — usuarios con patrón atípico vs normal:")
print(fraud_atipico_pct.round(1).to_string())

## 8. Análisis temporal — ¿a qué hora del día se usan más las conversaciones?

In [ ]:
df_convs["hora"]      = df_convs["date"].dt.hour
df_convs["mes"]       = df_convs["date"].dt.month
df_convs["dia_semana"] = df_convs["date"].dt.dayofweek  # 0=Lun … 6=Dom

DIA_MAP = {0: 'Lun', 1: 'Mar', 2: 'Mié', 3: 'Jue', 4: 'Vie', 5: 'Sáb', 6: 'Dom'}
MES_MAP = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
           7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

hora_counts    = df_convs["hora"].value_counts().sort_index()
dia_counts     = df_convs["dia_semana"].value_counts().sort_index()
mes_counts     = df_convs["mes"].value_counts().sort_index()

peak_hora = hora_counts.idxmax()
print(f"Hora pico de interacciones: {peak_hora:02d}:00 ({hora_counts[peak_hora]:,} msgs)")
print(f"Día pico: {DIA_MAP[dia_counts.idxmax()]} ({dia_counts.max():,} msgs)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4), facecolor=PALETTE['bg'])

# 1) Hora del día
ax = axes[0]
bar_colors_hora = [PALETTE['primary'] if h == peak_hora else PALETTE['secondary']
                   for h in hora_counts.index]
ax.bar(hora_counts.index, hora_counts.values, color=bar_colors_hora)
ax.set_title("Interacciones por Hora del Día", fontsize=11, color=PALETTE['text'])
ax.set_xlabel("Hora", color=PALETTE['text'])
ax.set_ylabel("N° de msgs", color=PALETTE['text'])
ax.set_xticks(range(0, 24, 3))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_facecolor(PALETTE['bg'])
ax.spines[['top','right']].set_visible(False)
ax.annotate(f"Pico:\n{peak_hora:02d}:00",
            xy=(peak_hora, hora_counts[peak_hora]),
            xytext=(peak_hora + 2, hora_counts[peak_hora] * 0.9),
            fontsize=8, color=PALETTE['primary'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['primary'], lw=1))

# 2) Día de la semana
ax2 = axes[1]
dias_labels = [DIA_MAP[d] for d in dia_counts.index]
bar_colors_dia = [PALETTE['accent'] if i >= 5 else PALETTE['secondary']
                  for i in dia_counts.index]
ax2.bar(dias_labels, dia_counts.values, color=bar_colors_dia)
ax2.set_title("Interacciones por Día de Semana", fontsize=11, color=PALETTE['text'])
ax2.set_xlabel("Día", color=PALETTE['text'])
ax2.set_ylabel("N° de msgs", color=PALETTE['text'])
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax2.set_facecolor(PALETTE['bg'])
ax2.spines[['top','right']].set_visible(False)

# 3) Evolución mensual
ax3 = axes[2]
mes_labels = [MES_MAP[m] for m in mes_counts.index]
ax3.plot(mes_labels, mes_counts.values,
         marker='o', color=PALETTE['secondary'], lw=2, markersize=5)
ax3.fill_between(range(len(mes_labels)), mes_counts.values,
                 alpha=0.15, color=PALETTE['secondary'])
ax3.set_title("Volumen Mensual de Interacciones", fontsize=11, color=PALETTE['text'])
ax3.set_xlabel("Mes", color=PALETTE['text'])
ax3.set_ylabel("N° de msgs", color=PALETTE['text'])
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax3.set_facecolor(PALETTE['bg'])
ax3.spines[['top','right']].set_visible(False)
ax3.set_xticks(range(len(mes_labels)))
ax3.set_xticklabels(mes_labels, rotation=30)

fig.suptitle("UC4 — Patrones Temporales de Conversaciones con Havi",
             fontsize=14, fontweight='bold', color=PALETTE['text'])
plt.tight_layout()
plt.savefig("uc4_fig7_temporal.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig7_temporal.png")

### 8.1 Heatmap hora × día

In [ ]:
pivot_hora_dia = df_convs.groupby(["hora", "dia_semana"]).size().unstack(fill_value=0)
pivot_hora_dia.columns = [DIA_MAP[d] for d in pivot_hora_dia.columns]

fig, ax = plt.subplots(figsize=(10, 6), facecolor=PALETTE['bg'])
im = ax.imshow(pivot_hora_dia.values, aspect='auto', cmap='YlOrRd', origin='upper')
ax.set_xticks(range(7))
ax.set_xticklabels(pivot_hora_dia.columns, color=PALETTE['text'])
ax.set_yticks(range(24))
ax.set_yticklabels([f"{h:02d}h" for h in range(24)], fontsize=7, color=PALETTE['text'])
ax.set_xlabel("Día de la semana", color=PALETTE['text'])
ax.set_ylabel("Hora del día", color=PALETTE['text'])
ax.set_title("Heatmap: Interacciones por Hora × Día de Semana",
             fontsize=13, fontweight='bold', color=PALETTE['text'])
plt.colorbar(im, ax=ax, label='N° interacciones')
ax.set_facecolor(PALETTE['bg'])
plt.tight_layout()
plt.savefig("uc4_fig8_heatmap_hora_dia.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: uc4_fig8_heatmap_hora_dia.png")

## 9. Resumen ejecutivo — Hallazgos clave UC4

In [ ]:
print("=" * 65)
print("  UC4 — INTELIGENCIA CONVERSACIONAL — HALLAZGOS CLAVE")
print("=" * 65)

print(f"""
── COBERTURA ──────────────────────────────────────────────────
  Turnos totales         : {n_turnos:,}
  Conversaciones únicas  : {n_convs:,}
  Usuarios únicos        : {n_users:,}  (100% cobertura de clientes)
  Período                : {fecha_min.date()} → {fecha_max.date()}

── CANAL ──────────────────────────────────────────────────────
  Texto (channel=1)      : {canal_pct.get('Texto', 0):.1f}%
  Voz   (channel=2)      : {canal_pct.get('Voz', 0):.1f}%

── CONVERSACIONES ─────────────────────────────────────────────
  Turnos / conv (media)  : {media_turnos:.2f}  (mayoría = 1-2 turnos)
  Input — media palabras : {df_convs['input_tokens'].mean():.1f}
  Output — media palabras: {df_convs['output_tokens'].mean():.1f}

── INTENCIONES (clasificador keywords) ────────────────────────""")

for intent in intent_counts.index:
    print(f"  {intent:<25}: {intent_counts[intent]:>6,}  ({intent_pct[intent]:.1f}%)")

print(f"""
── TEMPORAL ───────────────────────────────────────────────────
  Hora pico              : {peak_hora:02d}:00
  Día pico               : {DIA_MAP[dia_counts.idxmax()]}

── VALIDACIÓN UCs ──────────────────────────────────────────────
  UC1 (problema_transaccion + fraude)  → usuarios YA reportan estos problemas
  UC3 (solicitud_producto)             → intención de adquirir productos detectada
  UC4 (intenciones clasificadas)       → base para NLP de producción
""")
print("=" * 65)

In [ ]:
# ── Exportar taxonomía de intents para UC4 ────────────────────────────────────
from pathlib import Path
import json

intent_taxonomy = {
    "total_turnos": int(n_turnos),
    "total_conversaciones": int(n_convs),
    "usuarios_unicos": int(n_users),
    "periodo": {"inicio": str(fecha_min.date()), "fin": str(fecha_max.date())},
    "canal": {k: int(v) for k, v in canal_counts.items()},
    "intents": {
        intent: {"count": int(intent_counts[intent]), "pct": float(intent_pct[intent])}
        for intent in intent_counts.index
    },
    "tokens_promedio": {
        "input": round(float(df_convs['input_tokens'].mean()), 1),
        "output": round(float(df_convs['output_tokens'].mean()), 1)
    },
    "hora_pico": int(peak_hora),
    "top20_palabras": {word: int(count) for word, count in top20}
}

out_path = Path("outputs")
out_path.mkdir(exist_ok=True)
out_file = out_path / "uc4_intent_taxonomy.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(intent_taxonomy, f, ensure_ascii=False, indent=2)

print(f"Taxonomía exportada a: {out_file}")
print(json.dumps(intent_taxonomy, ensure_ascii=False, indent=2)[:800], "...")